In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from functools import reduce
xr.set_options(display_max_rows=9999)

In [ ]:
VAR = 'Theta_0'  # any pred var: Theta_k, Salt_k, U_k, V_k (k=0-50), Eta; coarse preds only have level 0 and Eta

R = '/orcd/scratch/seedfund/001/llc_emulator/inference/multiscale/2026-11-15:multiscale_llc:'
EMULATORS = {  # display name -> inference dir holding sharp_pred.zarr and coarse_pred.zarr; llc is always panel 1
    'warm ep11':             R + 'face1-warm-lr1.5e-4/ema_ckpt_ep0011_2012-10-14T00_2012-11-14T00',
    # 'warm ep11 f32':         R + 'face1-warm-lr1.5e-4/ema_ckpt_ep0011_2012-10-14T00_2012-10-15T00_f32',
    # 'warm ep11 f16 shapiro': R + 'face1-warm-lr1.5e-4/ema_ckpt_ep0011_2012-10-14T00_2012-10-15T00_f16-shapiro4-0.1',
    'warm ep11 f32 shapiro': '/orcd/scratch/seedfund/001/llc_emulator/multiscale/2026-11-15:multiscale_llc:face1-warm-lr1.5e-4/ema_ckpt_ep0011_2012-10-14T00_2012-11-14T00',
    # 'cold ep18':           R + 'face1-cold-channelnorm-lr4e-4/ema_ckpt_ep0018_2012-10-14T00_2012-11-14T00',
    # 'cold ecmwf ep6':       '/orcd/scratch/seedfund/001/llc_emulator/multiscale/2026-11-20:multiscale_llc:face1-cold-ecmwf-log1p/ema_ckpt_ep0006_2012-10-14T00_2012-10-16T00' ,
    # 'cold ep6': '/orcd/scratch/seedfund/001/llc_emulator/multiscale//2026-11-15:multiscale_llc:face1-cold-channelnorm-lr4e-4/ema_ckpt_ep0006_2012-10-14T00_2012-10-16T00',
    # 'cold ep5': '/orcd/scratch/seedfund/001/llc_emulator/multiscale//2026-11-15:multiscale_llc:face1-cold-channelnorm-lr4e-4/ema_ckpt_ep0005_2012-10-14T00_2012-10-16T00',
    #'fp32 dil ep 0.5': '/orcd/scratch/seedfund/001/llc_emulator/inference/multiscale/2026-11-22:multiscale_llc:face1-cold-ecmwf-log1p-fp32-raw-dil/ckpt_emergency_2012-10-14T00_2012-10-15T00'
}

BASE = VAR.split('_')[0]
CMAP = {'Theta': 'magma', 'Salt': 'viridis', 'U': 'RdBu_r', 'V': 'RdBu_r', 'Eta': 'cividis'}[BASE]

def limits(a):
    """Colour range from the truth field; symmetric about 0 for velocities."""
    if BASE in ('U', 'V'):
        m = np.nanmax(np.abs(a))
        return -m, m
    return np.nanmin(a), np.nanmax(a)

def panels(n, dpi=None, size=4.5):
    """Near-square grid for n panels (1x2, 2x2, 2x3, 3x3, ...), shared axes, spare axes hidden."""
    ncols = int(np.ceil(np.sqrt(n)))
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(size * ncols + 1, size * nrows), dpi=dpi,
                            sharex=True, sharey=True, constrained_layout=True, squeeze=False)
    axs = axs.ravel()
    for ax in axs[n:]:
        ax.set_visible(False)
    return fig, axs[:n], axs

def common_times(preds):
    """Timestamps every pred has, so time index i is the same hour in every panel."""
    return reduce(np.intersect1d, [p.time.values for p in preds])

In [ ]:
llc = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320', consolidated=False)

def llc_face1(name):
    """Raw LLC face 1 as the preds store it: lat/lon = j/i, land NaN, U/V centred and rotated east/north."""
    k = 0 if name == 'Eta' else int(name.split('_')[1])
    g = llc.isel(face=1, k=k)
    cen = lambda a: a.rename(j='lat', i='lon')
    if BASE in ('U', 'V'):  # same as multiscale_data.to_agrid: NaN->0, average with next, last kept, rotate
        u = g.U.rename(j='lat', i_g='lon').fillna(0)
        v = g.V.rename(j_g='lat', i='lon').fillna(0)
        u = (0.5 * (u + u.shift(lon=-1))).fillna(u)
        v = (0.5 * (v + v.shift(lat=-1))).fillna(v)
        cs, sn = cen(g.CS), cen(g.SN)
        da = u * cs - v * sn if BASE == 'U' else u * sn + v * cs
    else:
        da = cen(g[BASE])
    return da.where(cen(g.mask_c))

# pred lat/lon are face-1 j/i indices and pred times are target hours, so sel LLC on the shared pred times
sharp = {name: xr.open_zarr(f'{d}/sharp_pred.zarr')[VAR] for name, d in EMULATORS.items()}
t = common_times(sharp.values())
runs = {'llc': llc_face1(VAR).sel(time=t), **{name: p.sel(time=t) for name, p in sharp.items()}}
truth = runs['llc']
print(f'sharp: {len(t)} shared hours, {t[0]} to {t[-1]}')

def show(**sel):
    """llc + every emulator at the same isel, on the llc colour range."""
    f = [r.isel(**sel).values.astype(np.float32) for r in runs.values()]
    vmin, vmax = limits(f[0])
    fig, axs, every = panels(len(f))
    for ax, name, a in zip(axs, runs, f):
        im = ax.imshow(a, vmin=vmin, vmax=vmax, cmap=CMAP)
        ax.set_title(name)
    fig.colorbar(im, ax=every, shrink=0.9)
    fig.suptitle(f"{VAR}  {np.datetime_as_string(truth.time.values[sel['time']], unit='h')}")
    plt.show()

In [ ]:
import json

# cache is normalized by its native store's per-level mean/std and has no land NaNs, so undo both
# all 13 faces kept (pick one in sel_c); pred and cache share the layout, faces 7-12 rotated north-up, U/V east/north
cache = xr.open_zarr('/orcd/scratch/seedfund/001/llc_emulator/coarse/LLC4320_coarse_cache_0p33deg.zarr')
native = xr.open_zarr(cache.attrs['native_store'], consolidated=False)
ch = json.loads(cache.attrs['prognostic_json']).index(VAR)
k = 0 if VAR == 'Eta' else int(VAR.split('_')[1])
mu, sd = float(native[f'{BASE}_mean'][k]), float(native[f'{BASE}_std'][k])
wet = cache.wet.isel(channel=ch).rename(y='lat', x='lon').load().astype(bool)
truth_c = cache.prognostic.isel(channel=ch).rename(y='lat', x='lon').astype(np.float32).where(wet) * sd + mu

coarse = {name: xr.open_zarr(f'{d}/coarse_pred.zarr')[VAR] for name, d in EMULATORS.items()}
t_c = common_times(coarse.values())
runs_c = {'llc coarse': truth_c.sel(time=t_c), **{name: p.sel(time=t_c) for name, p in coarse.items()}}
print(f'coarse: {len(t_c)} shared steps, {t_c[0]} to {t_c[-1]}')

In [ ]:
from matplotlib import animation
from IPython.display import HTML
plt.rcParams['animation.embed_limit'] = 300  # MB; many panels x hundreds of frames

def video(runs, fps=24, dpi=70, **sel):
    """Inline video of every run in a near-square grid over sel (time a slice), on the first run's colour range."""
    da = [r.isel(**sel) for r in runs.values()]
    f = [d.values.astype(np.float32) for d in da]
    times = np.datetime_as_string(da[0].time.values, unit='h')
    vmin, vmax = limits(f[0])
    fig, axs, every = panels(len(f), dpi=dpi)
    ims = [ax.imshow(a[0], vmin=vmin, vmax=vmax, cmap=CMAP) for ax, a in zip(axs, f)]
    for ax, name in zip(axs, runs):
        ax.set_title(name)
    fig.colorbar(ims[0], ax=every, shrink=0.9)
    title = fig.suptitle(f'{VAR}  {times[0]}')

    def update(i):
        for im, a in zip(ims, f):
            im.set_data(a[i])
        title.set_text(f'{VAR}  {times[i]}')
        return *ims, title

    anim = animation.FuncAnimation(fig, update, frames=len(times))
    plt.close(fig)
    return HTML(anim.to_jshtml(fps=fps, default_mode='loop'))

In [ ]:
sel = dict(time=slice(0, 24*4), lat=slice(2160+300+25, 2160+300+75), lon=slice(720+300+25, 720+300+75))
video(runs, **sel)

In [ ]:
# sharp video: hourly, face-1 j/i indices
#sel = dict(time=slice(0, 24), lat=slice(j0+100, j0+300), lon=slice(i0+100, i0+300))
#sel = dict(time=slice(0, 24), lat=slice(2160, 2880), lon=slice(720, 1440))
sel = dict(time=slice(0, 24*4), lat=slice(0,200), lon=slice(4320-200, 4320))
video(runs, **sel)

In [ ]:
# sharp video: hourly, face-1 j/i indices
#sel = dict(time=slice(0, 24), lat=slice(j0+100, j0+300), lon=slice(i0+100, i0+300))
#sel = dict(time=slice(0, 24), lat=slice(2160, 2880), lon=slice(720, 1440))
sel = dict(time=slice(0, 24*4), lat=slice(720,1440), lon=slice(2880, 3600))
video(runs, **sel)

In [ ]:
sel = dict(time=slice(0, 24*4), lat=slice(720,720+50), lon=slice(2880, 2880+50))
video(runs, **sel)

In [ ]:
sel = dict(time=slice(0, 24*31), lat=slice(3600+100,4320-100), lon=slice(720, 1440-200))
video(runs, **sel)

In [ ]:
# coarse video: 3-hourly (time=8 steps per day), face 0-12, coarse index = sharp index // 16
sel_c = dict(time=slice(0, 248), face=1, lat=slice(2160//16, 2880//16), lon=slice(720//16, 1440//16))
video(runs_c, **sel_c)

In [ ]:
show(time=T, lat=slice(j0, j0+720), lon=slice(i0, i0+720))

In [ ]:
show(time=T, lat=slice(j0+100, j0+300), lon=slice(i0+100, i0+300))

In [ ]:
show(time=T, lat=slice(j0+100+60, j0+100+80), lon=slice(i0+100+55, i0+100+75))

In [ ]:
# agulhas comparison
show(time=T, lat=slice(720, 1440), lon=slice(2880, 3600))
show(time=T, lat=slice(720, 720+20), lon=slice(2880+500, 2880+500+20))